# Day 11 Tutorial：用 Pipeline 防止预处理泄漏

## Goal

在含缺失值和验证分布变化的人工数据上，对照全表预处理的错误统计量与只在训练数据拟合的 Pipeline，并检查 Pipeline 内部状态。


## Setup

本教程故意让验证特征的一列发生平移，使“全表拟合”与“训练拟合”的缩放均值明显不同。分数高低不是泄漏判定标准；信息边界才是。


In [ ]:
import platform
import numpy as np
import pandas as pd
import sklearn
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
print({"python": platform.python_version(), "sklearn": sklearn.__version__})


## Steps

### 1. 先固定训练与验证，再加入缺失值


In [ ]:
X = rng.normal(size=(80, 4))
X[60:, 0] += 3.0  # 只用于教学：验证区域该特征发生分布平移
y = 2.0 * X[:, 0] - 1.5 * X[:, 1] + 0.7 * X[:, 2] + rng.normal(0, 0.3, size=80)

missing_rows = np.array([3, 12, 25, 44, 63, 71])
missing_columns = np.array([0, 2, 1, 3, 0, 2])
X[missing_rows, missing_columns] = np.nan

X_train, X_valid = X[:60], X[60:]
y_train, y_valid = y[:60], y[60:]
print({"train": X_train.shape, "validation": X_valid.shape, "missing": int(np.isnan(X).sum())})


### 2. 只展示错误流程学到的统计量

下面故意在训练+验证上拟合预处理器，仅用于审计对照；不把它作为合格模型结果。


In [ ]:
wrong_imputer = SimpleImputer(strategy="median")
X_all_imputed_wrong = wrong_imputer.fit_transform(X)
wrong_scaler = StandardScaler().fit(X_all_imputed_wrong)

wrong_statistics = pd.DataFrame({
    "feature": np.arange(X.shape[1]),
    "all_data_scaler_mean_WRONG": wrong_scaler.mean_,
})
display(wrong_statistics.round(4))


### 3. 正确 Pipeline 只在训练区拟合


In [ ]:
ridge_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0)),
])
ridge_pipeline.fit(X_train, y_train)

train_prediction = ridge_pipeline.predict(X_train)
valid_prediction = ridge_pipeline.predict(X_valid)

def metrics_row(split, truth, prediction):
    return {
        "split": split,
        "mae": float(mean_absolute_error(truth, prediction)),
        "rmse": float(root_mean_squared_error(truth, prediction)),
        "r2": float(r2_score(truth, prediction)),
    }

metrics = pd.DataFrame([
    metrics_row("train", y_train, train_prediction),
    metrics_row("validation", y_valid, valid_prediction),
])
display(metrics.round(4))


### 4. 比较错误与正确的内部统计量

正确 scaler 的均值应等于“训练数据经训练 imputer 填补后”的列均值。


In [ ]:
fitted_imputer = ridge_pipeline.named_steps["imputer"]
fitted_scaler = ridge_pipeline.named_steps["scaler"]
train_imputed = fitted_imputer.transform(X_train)
expected_train_mean = train_imputed.mean(axis=0)

statistics_comparison = pd.DataFrame({
    "feature": np.arange(X.shape[1]),
    "training_only_mean_CORRECT": fitted_scaler.mean_,
    "all_data_mean_WRONG": wrong_scaler.mean_,
    "difference": wrong_scaler.mean_ - fitted_scaler.mean_,
})
display(statistics_comparison.round(4))


## Checks

检查正确 Pipeline 的状态来源、数组形状和数值有效性。


In [ ]:
assert list(ridge_pipeline.named_steps) == ["imputer", "scaler", "ridge"]
assert np.allclose(fitted_imputer.statistics_, np.nanmedian(X_train, axis=0))
assert np.allclose(fitted_scaler.mean_, expected_train_mean)
assert not np.allclose(fitted_scaler.mean_, wrong_scaler.mean_)
assert X_train.shape[1] == X_valid.shape[1]
assert np.isfinite(metrics[["mae", "rmse", "r2"]]).all().all()

print("Checks passed: preprocessing state matches training-only statistics.")


## Next Steps

完成练习后，把整条 Pipeline 交给交叉验证或参数搜索，使每一折都重新拟合预处理。Pipeline 不会自动判断分组、未来特征或科学语义；这些仍需研究者审查。
